# 9-bob. Tarmoq hujumlarini aniqlash: SI asosidagi bostirib kirishni aniqlash tizimlari
## Kengaytirilgan amaliy ish: uch tashkilot uchun federativ hujumlarni aniqlash modeli

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/09-bob/9-3_uch-tashkilot-uchun-federativ-hujumlarni.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 9-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dasturda uchta tashkilot oqim xususiyatlari asosida hujumlarni aniqlaydi, lekin har biri hujumlarning faqat bir qismini ko'rgan: bank — DDoS va parol tanlashni, universitet — skanerlashni, shifoxona — ma'lumotlarni chiqarib yuborishni. Model sifatida NumPy yordamida yozilgan logistik regressiya ishlatiladi. Uchta yondashuv taqqoslanadi: har bir tashkilotning faqat o'z ma'lumotlarida o'qitilgan modeli, 10 raundli FedAvg va barcha ma'lumotlar bir joyga to'plangan markazlashgan model. Har bir model barcha to'rt hujum turida va odatiy trafikda sinaladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(9)

In [ ]:
# 1. Oqim xususiyatlari: davomiylik, paketlar, baytlar, turli portlar, SYN ulushi, chiquvchi/kiruvchi baytlar
def oqimlar(tur, n):
    X = rng.normal([5, 20, 9, 2, 0.1, 0.5], [2, 6, 1.5, 1, 0.05, 0.2], (n, 6))   # oddiy trafik
    if tur == "ddos":     X[:, [1, 4]] += [80, 0.7]           # ko'p paket, SYN ko'p
    if tur == "skaner":   X[:, [0, 3]] += [-4, 40]           # qisqa, ko'p turli portlar
    if tur == "parol":    X[:, [0, 1]] += [-3, 15]           # qisqa takroriy ulanishlar
    if tur == "chiqarish": X[:, [2, 5]] += [5, 4]            # katta hajmdagi chiquvchi ma'lumot
    return X

def tashkilot(hujumlar, n_oddiy=3000, n_hujum=300):
    X = np.vstack([oqimlar("oddiy", n_oddiy)] + [oqimlar(h, n_hujum) for h in hujumlar])
    y = np.r_[np.zeros(n_oddiy), np.ones(n_hujum * len(hujumlar))]
    return X, y

# Har bir tashkilot faqat o'zi duch kelgan hujumlarni ko'rgan
tashkilotlar = {"bank": tashkilot(["ddos", "parol"]),
                "universitet": tashkilot(["skaner"]),
                "shifoxona": tashkilot(["chiqarish"])}
masshtab = StandardScaler().fit(oqimlar("oddiy", 5000))      # umumiy kelishilgan masshtablash

def oqitish(X, y, w, qadam=200, lr=0.5):
    """Logistik regressiyani gradiyent tushish bilan o'qitish (w oxirgi elementi - siljish)."""
    Xb = np.c_[X, np.ones(len(X))]
    vazn = np.where(y == 1, len(y) / (2 * y.sum()), len(y) / (2 * (len(y) - y.sum())))  # sinflarga vazn
    for _ in range(qadam):
        p = 1 / (1 + np.exp(-Xb @ w))
        w = w - lr * (Xb.T @ (vazn * (p - y))) / len(y)
    return w

def bashorat(w, X):
    return (np.c_[X, np.ones(len(X))] @ w > 0).astype(int)

In [ ]:
# 2. Federativ o'qitish (FedAvg): har raundda mahalliy o'qitish, so'ng faqat vaznlar o'rtachalanadi
w_global = np.zeros(7)
for raund in range(10):
    yangilar, hajmlar = [], []
    for X, y in tashkilotlar.values():
        yangilar.append(oqitish(masshtab.transform(X), y, w_global.copy(), qadam=20))
        hajmlar.append(len(y))
    w_global = np.average(yangilar, axis=0, weights=hajmlar)

In [4]:
# 3. Taqqoslash: mahalliy modellar va barcha ma'lumotlar bir joyga to'plangan markazlashgan model
mahalliy = {nom: oqitish(masshtab.transform(X), y, np.zeros(7)) for nom, (X, y) in tashkilotlar.items()}
X_hammasi = np.vstack([X for X, _ in tashkilotlar.values()])
y_hammasi = np.concatenate([y for _, y in tashkilotlar.values()])
markaz = oqitish(masshtab.transform(X_hammasi), y_hammasi, np.zeros(7))

print(f"{'Model':<24}" + "".join(f"{h:>11}" for h in ("ddos", "parol", "skaner", "chiqarish")) + "   noto'g'ri signal")
X_oddiy = masshtab.transform(oqimlar("oddiy", 5000))
modellar = {**{f"faqat {k}": v for k, v in mahalliy.items()}, "federativ": w_global, "markazlashgan": markaz}
for nom, w in modellar.items():
    aniqlash = [bashorat(w, masshtab.transform(oqimlar(h, 1000))).mean() for h in ("ddos", "parol", "skaner", "chiqarish")]
    print(f"{nom:<24}" + "".join(f"{a:>11.2f}" for a in aniqlash) + f"{bashorat(w, X_oddiy).mean():>12.3f}")

Model                          ddos      parol     skaner  chiqarish   noto'g'ri signal
faqat bank                     1.00       0.91       0.00       0.30       0.055
faqat universitet              0.00       0.00       1.00       0.00       0.000
faqat shifoxona                0.00       0.00       0.00       1.00       0.000
federativ                      1.00       0.62       1.00       0.99       0.006
markazlashgan                  1.00       0.80       1.00       1.00       0.035


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Model                          ddos      parol     skaner  chiqarish   noto'g'ri signal
faqat bank                     1.00       0.91       0.00       0.30       0.055
faqat universitet              0.00       0.00       1.00       0.00       0.000
faqat shifoxona                0.00       0.00       0.00       1.00       0.000
federativ                      1.00       0.62       1.00       0.99       0.006
markazlashgan                  1.00       0.80       1.00       1.00       0.035
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 9-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.